# SecureCode AI M-A2026 reproducible demonstration

This notebook runs the pinned public CWE-89 composition in an ephemeral workspace, summarizes the recorded 312-cell development experiment, and shows the redacted real-local model receipt. It does not execute development-corpus source or contact a model provider.


In [1]:
import json
import sys
import tempfile
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
{'repository': ROOT.name, 'execution_boundary': 'pinned public fixtures only'}


{'repository': 'm-a2026', 'execution_boundary': 'pinned public fixtures only'}

## Pinned audit composition

The demonstration scans one vulnerable and one safe synthetic fixture, proposes a reference repair, validates an ephemeral fixed copy, and confirms that the original fixture is unchanged.


In [2]:
from demo.mvp_cwe89_demo import run_demo

with tempfile.TemporaryDirectory(prefix='m-a2026-audit-') as temporary:
    demo_manifest = run_demo(Path(temporary) / 'demo')

audit_receipt = {
    'reference_outcome': demo_manifest['reference_outcome'],
    'signal_counts': demo_manifest['signal_counts'],
    'patch_proposed': bool(demo_manifest['patch_sha256']),
    'original_checkout_changed': False,
    'network_access': demo_manifest['network_access'],
    'product_pass_claimed': demo_manifest['product_pass'],
}
audit_receipt


{'reference_outcome': 'COMPLETED', 'signal_counts': {'vulnerable': 1, 'safe_control': 0, 'ephemeral_fixed': 0}, 'patch_proposed': True, 'original_checkout_changed': False, 'network_access': 'not_used', 'product_pass_claimed': False}

## Recorded development experiment

These values are read from the recomputed aggregate. Failed model cells remain failures; the notebook does not turn an incomplete run into a clean result.


In [3]:
aggregate = json.loads((ROOT / 'evaluation/development/results/aggregate.json').read_text(encoding='utf-8'))
configurations = aggregate['configurations']
experiment_receipt = {
    'planned_cells': aggregate['planned_cells'],
    'recorded_cells': aggregate['recorded_cells'],
    'failed_cells': sum(item['failed'] for item in configurations.values()),
    'incomplete': aggregate['incomplete'],
    'deterministic_only': {key: configurations['deterministic_only'][key] for key in ('TP', 'FP', 'TN', 'FN', 'precision', 'recall', 'f1')},
    'one_shot_llm': {key: configurations['one_shot_llm'][key] for key in ('TP', 'FP', 'TN', 'FN', 'precision', 'recall', 'f1', 'failed')},
    'repair_metrics': aggregate['repair_metrics'],
}
experiment_receipt


{'planned_cells': 312, 'recorded_cells': 312, 'failed_cells': 171, 'incomplete': True, 'deterministic_only': {'TP': 1, 'FP': 0, 'TN': 12, 'FN': 11, 'precision': 1.0, 'recall': 0.08333333333333333, 'f1': 0.15384615384615385}, 'one_shot_llm': {'TP': 9, 'FP': 3, 'TN': 24, 'FN': 27, 'precision': 0.75, 'recall': 0.25, 'f1': 0.375, 'failed': 24}, 'repair_metrics': {'attempted_patches': 0, 'correct_and_secure_rate': None, 'independently_validated_patches': 0, 'root_cause_repair_rate': None, 'sandbox_validated_patches': 0}}

## Redacted real-local model evidence

The retained receipt records observed runtime identity, lane agreement, patch status, and ephemeral validation. Raw source, prompt, model response, and patch bytes are excluded.


In [4]:
local_receipt = json.loads((ROOT / 'report/m-a2026/evidence/p917-real-local/integrator-receipt.json').read_text(encoding='utf-8'))
real_local_summary = {
    'outcome': local_receipt['outcome'],
    'lane_agreement': local_receipt['lane_agreement'],
    'patch_status': local_receipt['patch_status'],
    'ephemeral_validation_status': local_receipt['ephemeral_validation_status'],
    'source_unchanged': local_receipt['source_unchanged'],
    'runtime_identity': local_receipt['runtime_identity'],
}
real_local_summary


{'outcome': 'COMPLETED', 'lane_agreement': 'AGREED', 'patch_status': 'PROPOSED', 'ephemeral_validation_status': 'PASSED', 'source_unchanged': True, 'runtime_identity': {'endpoint': 'http://127.0.0.1:11434/v1', 'identity_status': 'OBSERVED', 'model_digest': 'sha256:dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e75ed87571486f4364', 'model_id': 'qwen2.5-coder:7b-instruct-q4_K_M', 'observation_boundary': 'before_and_after_calls', 'observation_source': 'literal_loopback_version_and_tags', 'profile_id': 'local-source-model', 'quantization': 'Q4_K_M', 'runtime_version': '0.16.2'}}

## Interpretation

The pinned composition demonstrates the intended detect, suggest, ephemeral-validate, and report flow for one Python CWE-89 case. The development study remains incomplete: 171 model cells failed and no patch entered the repair study. These results support an academic prototype demonstration, not a general accuracy, production, or release-readiness claim.
